# Les files — Exercice 1 : la classe `File`

**Notions travaillées :** structure de données *file* (FIFO), programmation objet, implémenter une API donnée, gérer une erreur, méthode spéciale `__repr__`.

> **Rappel de cours**
> - Une **file** fonctionne en mode **FIFO** (*First In, First Out*) : le premier élément entré est le premier sorti, comme dans une **file d'attente**.
> - `enfile(el)` ajoute `el` **à la fin** (en queue) ; `defile()` retire et renvoie l'élément **en tête**.
> - À comparer avec la **pile** (LIFO, *Last In, First Out*) : on retire le **dernier** élément ajouté, comme une pile d'assiettes.

## Question 1 : écrire la classe `File` dans un fichier `file.py`

### Le choix de représentation

On range les éléments dans une liste python `self.contenu` (c'est autorisé **uniquement dans cet exercice**) :
- la **tête** de la file est au **début** de la liste (indice 0) ;
- la **queue** est à la **fin** de la liste.

| Méthode | Comment on la code | Pourquoi |
|:---|:---|:---|
| `enfile(el)` | `self.contenu.append(el)` | `append` ajoute à la fin de la liste, donc en queue |
| `defile()` | `self.contenu.pop(0)` | `pop(i)` **supprime et renvoie** l'élément d'indice `i` : `pop(0)` renvoie la tête |
| `taille()` | `len(self.contenu)` | |
| `est_vide()` | `self.taille() == 0` | on réutilise une méthode déjà écrite |

💡 **L'indication de l'énoncé :** la classe `Pile` utilise `pop()` **sans argument**, qui retire le **dernier** élément (LIFO). Pour une file, on retire le **premier** : c'est `pop(0)`. Toute la différence entre pile et file tient dans cet indice !

### « Déclenche une erreur si la file est vide »

Si la file est vide, `pop(0)` déclencherait déjà une erreur, mais avec un message obscur (`pop from empty list`) qui parle d'une liste alors que l'utilisateur manipule une file. On déclenche donc **nous-mêmes** une erreur claire avec `raise` :
```python
raise IndexError("defile() sur une file vide")
```

### `__repr__`

C'est une **méthode spéciale** : Python l'appelle automatiquement quand on affiche l'objet (`print(f)` ou simplement `f` en fin de cellule). Sans elle, on verrait `<file.File object at 0x7f...>`. On choisit d'afficher `tête < ... < queue` pour bien voir **par où on défile** (à gauche) et **par où on enfile** (à droite).

La cellule ci-dessous **écrit le fichier** `file.py` (commande `%%writefile` de Jupyter), puis on l'importe.

In [1]:
%%writefile file.py
class File:
    """File (queue) en mode FIFO : premier entré, premier sorti.

    Les éléments sont rangés dans une liste python : la tête de la file
    est au début de la liste (indice 0), la queue à la fin.
    """

    def __init__(self):
        """Construit une nouvelle file vide."""
        self.contenu = []

    def enfile(self, el):
        """Ajoute un élément à la fin de la file."""
        self.contenu.append(el)

    def defile(self):
        """Supprime et renvoie l'élément en tête de file.
        Déclenche une erreur si la file est vide."""
        if self.est_vide():
            raise IndexError("defile() sur une file vide")
        return self.contenu.pop(0)

    def taille(self):
        """Renvoie le nombre d'éléments de la file."""
        return len(self.contenu)

    def est_vide(self):
        """Renvoie True si la file est vide, False sinon."""
        return self.taille() == 0

    def __repr__(self):
        """Renvoie une description du contenu de la file."""
        if self.est_vide():
            return "File vide"
        return "tête < " + " < ".join(repr(el) for el in self.contenu) + " < queue"

Overwriting file.py


In [2]:
from file import File

f = File()
print(f, "| taille :", f.taille(), "| vide ?", f.est_vide())
f.enfile(1)
f.enfile(2)
f.enfile(3)
print(f, "| taille :", f.taille(), "| vide ?", f.est_vide())
print("defile() renvoie", f.defile())
print(f)

File vide | taille : 0 | vide ? True
tête < 1 < 2 < 3 < queue | taille : 3 | vide ? False
defile() renvoie 1
tête < 2 < 3 < queue


### Tests

On vérifie le **FIFO** et le **cas d'erreur** avec `assert`. Pour tester qu'une erreur est bien déclenchée, on utilise `try / except`.

In [3]:
f = File()
assert f.est_vide() and f.taille() == 0
for x in [10, 20, 30]:
    f.enfile(x)
assert f.taille() == 3 and not f.est_vide()
assert f.defile() == 10          # le premier entré est le premier sorti
assert f.defile() == 20
assert f.defile() == 30
assert f.est_vide()

try:
    f.defile()
    print("ERREUR : aucune exception déclenchée")
except IndexError as e:
    print("Erreur bien déclenchée :", e)
print("Tous les tests passent ✔")

Erreur bien déclenchée : defile() sur une file vide
Tous les tests passent ✔


⚠️ **Erreurs fréquentes**
- Écrire `pop()` au lieu de `pop(0)` : on obtient une **pile**, pas une file.
- Oublier le `return` dans `defile` : l'élément est bien supprimé, mais la méthode renvoie `None`.
- Dans `__repr__`, utiliser `print` au lieu de `return` : `__repr__` doit **renvoyer** une chaîne.

💡 **Pour aller plus loin :** `pop(0)` doit décaler tous les éléments de la liste d'une case : son coût est proportionnel à la taille de la file. Le module `collections` propose `deque`, dont la méthode `popleft()` est en temps constant (hors programme).

---
## Question 2 : état de la file après le script

On suit le script **ligne par ligne** dans un tableau (la tête est à gauche) :

| Instruction | État de `f` | `x` |
|:---|:---|:---|
| `f = File()` | *(vide)* | |
| `f.enfile("n")` | `n` | |
| `f.enfile("s")` | `n s` | |
| `f.enfile("i")` | `n s i` | |
| `x = f.defile()` | `s i` | `"n"` |
| `f.defile()` | `i` | `"n"` |
| `f.enfile(x)` | `i n` | `"n"` |

**Réponse :** la file contient `"i"` en tête puis `"n"` en queue.

💡 La ligne `f.defile()` (sans affectation) supprime bien `"s"` : la valeur renvoyée est simplement perdue.

In [4]:
f = File()
f.enfile("n")
f.enfile("s")
f.enfile("i")
x = f.defile()
f.defile()
f.enfile(x)
print(f)

tête < 'i' < 'n' < queue
